# DER-only R-GCN variants — isolated rerun

This notebook reruns **only the nine DER-only R-GCN variants**. It does not train or overwrite Majority Class, Identity Prior, LR frames/triggers, LR bag-of-types, LR bag-of-nodes, or either sentence-only MLP. Keep the existing results for those methods separately and merge the newly generated R-GCN results into the comparison table after evaluation.

**R-GCN input constraint:** no Sentence node, full-sentence embedding, or sentence-contextualized span embedding. Each graph contains only annotated DER nodes and DER-to-DER relations. SBERT and BERT-ISO encode each DER span independently. The classifier uses graph-level mean pooling over DER nodes.

Upload the four annotation JSON files (`indibias_pos.json`, `indibias_neg.json`, `indicasa_pos.json`, `indicasa_neg.json`) to `/content` before running in Colab. For a comparable rerun, use the same stratified evaluation plans and seeds as the earlier experiment (see README for why "grouped" was dropped).


## 0. Setup & configuration


In [ ]:
# Install dependencies (Colab). spaCy is optional (identity lemmatiser falls back to plural stripping).
%pip install -q transformers sentence-transformers torch-geometric scikit-learn pandas numpy networkx matplotlib


In [ ]:
import copy, random, re, itertools, json, warnings, gc, time, os, hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import RGCNConv

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)

# ═════════════════════════ RUN PROFILE ═════════════════════════
RUN_PROFILE = "full"      # "quick": 1 repeat, InDiBias analysis only (sanity pass)
                           # "full" : 3 repeats, both datasets analysed (paper numbers)
PROFILES = {
    "quick": dict(n_repeats=1, abl_repeats=1, n_boot=1000, n_random_del=10,
                  analysis_plan=[("InDiBias", "stratified")]),
    "full":  dict(n_repeats=3, abl_repeats=1, n_boot=5000, n_random_del=20,
                  analysis_plan=[("InDiBias", "stratified"), ("IndiCasa", "stratified")]),
}
P = PROFILES[RUN_PROFILE]

SEED = 42
N_REPEATS   = P["n_repeats"]
ABL_REPEATS = P["abl_repeats"]
N_BOOT      = P["n_boot"]
N_RANDOM_DEL = P["n_random_del"]
SEEDS = [SEED + 100 * r for r in range(N_REPEATS)]
ABL_SEEDS = SEEDS[:ABL_REPEATS]

# Same evaluation plan for every variant. "grouped" CV was intended to protect InDiBias minimal
# pairs via template_id, but the annotation files carry no template_id (verified): every record
# gets its own group, so "grouped" produces a different but not actually leak-protected split from
# "stratified". Dropped "grouped" entirely rather than report a split with no real benefit over
# plain stratified CV -- see README "Known gaps" for the decision record.
EVAL_PLAN = [("InDiBias", "stratified"),
             ("IndiCasa", "stratified")]
ANALYSIS_PLAN = P["analysis_plan"]         # plans that also get ablations + interpretability + qualitative
assert all(p in EVAL_PLAN for p in ANALYSIS_PLAN)

# ═════════════════════════ GPU POLICY ═════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_ON_GPU = True        # tiny R-GCNs: set False if you want the GPU free for something else
train_device = device if TRAIN_ON_GPU else torch.device("cpu")
ENCODER_FP16 = True        # encoders run in fp16 on CUDA (vectors are cast back to fp32 and cached on CPU)
ENC_BATCH = 64


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_report(tag=""):
    if device.type != "cuda":
        print(f"[gpu] {tag}: running on CPU"); return
    mb = lambda b: b / 2**20
    print(f"[gpu] {tag}: allocated {mb(torch.cuda.memory_allocated()):.0f} MB | "
          f"reserved {mb(torch.cuda.memory_reserved()):.0f} MB | peak {mb(torch.cuda.max_memory_allocated()):.0f} MB")


# Training configs (identical for every model that uses them)
BASE_CFG = dict(epochs=50, lr=1e-3, weight_decay=1e-4, batch_size=16, hidden=64, dropout=0.3, num_bases=None)
REG_CFG  = dict(BASE_CFG, hidden=32, dropout=0.5, weight_decay=1e-3, epochs=30, num_bases=4)

os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
print(f"profile={RUN_PROFILE} | repeats={N_REPEATS} | seeds={SEEDS} | device={device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
gpu_report("start")


## 1. Data
`pos` = stereotype (label 1), `neg` = anti-stereotype (label 0).
Each InDiBias annotation carries a `template_id`; pos[i] and neg[i] come from the same template (49/50 differ only in the identity term).

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/content")

with open(DATA_DIR / "indibias_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDIBIAS = json.load(f)

with open(DATA_DIR / "indibias_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDIBIAS = json.load(f)

with open(DATA_DIR / "indicasa_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDICASA = json.load(f)

with open(DATA_DIR / "indicasa_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDICASA = json.load(f)

In [ ]:
print("InDiBias positive:", len(EXAMPLE_ANNOTATIONS_POS_INDIBIAS))
print("InDiBias negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDIBIAS))
print("IndiCasa positive:", len(EXAMPLE_ANNOTATIONS_POS_INDICASA))
print("IndiCasa negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDICASA))

InDiBias positive: 243
InDiBias negative: 83
IndiCasa positive: 243
IndiCasa negative: 83


In [ ]:
DATASETS = {
    "InDiBias": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDIBIAS, neg=EXAMPLE_ANNOTATIONS_NEG_INDIBIAS),
    "IndiCasa": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDICASA, neg=EXAMPLE_ANNOTATIONS_NEG_INDICASA),
}
for name, d in DATASETS.items():
    print(f"{name}: {len(d['pos'])} stereotype / {len(d['neg'])} anti-stereotype")

InDiBias: 243 stereotype / 83 anti-stereotype
IndiCasa: 243 stereotype / 83 anti-stereotype


## 2. DER annotations → sentence graphs

In [ ]:
# Identity normalisation ("Hindus" -> "hindu", "OBC family" -> "obc", ...)
try:
    import spacy
    _nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    _nlp = None          # falls back to simple plural stripping
print("spaCy lemmatiser:", "on" if _nlp is not None else "off (fallback)")

_IDENTITY_MODIFIERS = frozenset({
    "family", "families", "group", "community", "commnunity",
    "people", "person", "man", "woman", "child", "children"
})
_PLURAL_OVERRIDES = {
    "hindus": "hindu", "muslims": "muslim", "vaishyas": "vaishya", "brahmins": "brahmin",
    "dalits": "dalit", "obcs": "obc", "christians": "christian", "sikhs": "sikh",
    "jains": "jain", "buddhists": "buddhist",
}
_COMPOUND_PATTERNS = [
    (r"\b(obc|hindu|muslim|brahmin|dalit|vaishya|christian|sikh|jain)\s+(family|group|community|people)\b", r"\1"),
    (r"\b(hindu|muslim|brahmin|dalit|vaishya)\s+(man|woman|person|child)\b", r"\1"),
]


def _strip_plural(text):
    return text[:-1] if text.endswith("s") and not text.endswith(("ss", "us", "is")) else text


def _normalize_identity(span: str) -> str:
    text = span.lower().strip()
    for pattern, repl in _COMPOUND_PATTERNS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    words = text.split()
    if len(words) > 1 and words[-1] in _IDENTITY_MODIFIERS:
        text = " ".join(words[:-1])
    if text in _PLURAL_OVERRIDES:
        text = _PLURAL_OVERRIDES[text]
    if _nlp is not None:
        try:
            text = " ".join(t.lemma_ for t in _nlp(text))
        except Exception:
            text = _strip_plural(text)
    else:
        text = _strip_plural(text)
    return text.strip()

spaCy lemmatiser: on


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MAPPING IDENTITY TO CATEGORY
# ─────────────────────────────────────────────────────────────────────────────

_IDENTITY_TO_CATEGORY = {
    # Religion
    "hindu": "Religion",
    "muslim": "Religion",
    "christian": "Religion",
    "sikh": "Religion",
    "jain": "Religion",
    "buddhist": "Religion",
    "islam" :  "Religion",
    "christianity": "Religion",
    "hinduism": "Religion",
    "sikhism" : "Religion",
    "parsi" : "Religion",
    "parsis": "Religion",
    "buddhism": "Religion",
    "jainism": "Religion",

    # Caste
    "brahmin": "Caste",
    "vaishya": "Caste",
    "dalit": "Caste",
    "obc": "Caste",
    "sc": "Caste",
    "st": "Caste",
    "kshatriya": "Caste",
    "shudra": "Caste",
    "sc / st": "Caste",
    "kaystha": "Caste",
    "bania": "Caste",
    "kurmi": "Caste",
    "jat": "Caste",
    "rajput": "Caste",
    "adivasi": "Caste",
    "kayast": "Caste",
    "general": "Caste",
    "forward class": "Caste",
    "artisan": "Caste",
    "merchant": "Caste",
    "trader": "Caste",
    "laborer": "Caste",
    "peasant": "Caste",

    # Gender
    "women": "Gender",
    "man": "Gender",
    "woman": "Gender",
    "female": "Gender",
    "male": "Gender",
    "boy": "Gender",
    "girl": "Gender",
    "girls": "Gender",
    "boys": "Gender",
    "men": "Gender",
    "transgender male": "Gender",
    "lesbian": "Gender"
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. RELATION EXTRACTION (unchanged structure)
# ══════════════════════════════════════════════════════════════════════════════

class RuleBasedRelations:
    """
    Converts a DER annotation dict into a flat list of canonical edges.
    Structure unchanged, only the node extraction uses new normalization.
    """

    # ── Node-type labels ──────────────────────────────────────────────────
    IDENTITY_NODE   = "Identity Dimensions"
    PROFESSION_NODE = "Profession"
    TRIGGER_NODE    = "Contextual Trigger"
    FRAME_NODE      = "Situational Frame"

    # ── Canonical relation-type vocabulary ────────────────────────────────
    REL_IDENTITY_INTERSECTION        = "identity_intersection"
    REL_IDENTITY_ROLE_ASSOCIATION    = "identity_role_association"
    REL_CHARACTERIZED_AS             = "characterized_as"
    REL_IDENTITY_FRAME_ASSOCIATION= "identity_frame_association"

    # ── Copula / auxiliary / modal verbs ──────────────────────────────────
    PREDICATE_LEAD_WORDS: frozenset = frozenset({
        "is", "are", "was", "were", "be", "been", "being",
        "do", "does", "did",
        "has", "have", "had",
        "can", "could", "will", "would", "shall", "should",
        "may", "might", "must",
        "seem", "seems", "seemed",
        "appear", "appears", "appeared",
        "become", "becomes", "became",
        "get", "gets", "got",
        "remain", "remains", "remained",
    })

    @staticmethod
    def _clean(span: Any) -> str:
        return span.strip() if isinstance(span, str) else ""

    @staticmethod
    def _normalize(span: str) -> str:
        return _normalize_identity(span)

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 1 — Node extraction
    # ─────────────────────────────────────────────────────────────────────────

    def extract_nodes(self, annotation: Dict[str, Any]) -> Dict[str, List[Any]]:
        """
        Bucket the flat ``annotations`` list into typed groups.
        Now normalizes identity spans properly.
        """
        identities:  List[Dict[str, Any]] = []
        professions: List[str] = []
        triggers:    List[str] = []
        frames:      List[str] = []

        for ann in annotation.get("annotations", []):
            raw   = self._clean(ann.get("span", ""))
            if not raw:
                continue

            ntype = ann.get("node")

            # Only normalize identity nodes
            if ntype == self.IDENTITY_NODE:
                span = self._normalize(raw)
            else:
                span = raw.lower().strip()

            if ntype == self.IDENTITY_NODE:
                identities.append({
                    "span": span,
                    "label": ann.get("label"),
                    "category": _IDENTITY_TO_CATEGORY.get(span, "Other")
                })
            elif ntype == self.PROFESSION_NODE:
                professions.append(span)
            elif ntype == self.TRIGGER_NODE:
                triggers.append(span)
            elif ntype == self.FRAME_NODE:
                frames.append(span)

        return {
            "identities":  identities,
            "professions": professions,
            "triggers":    triggers,
            "frames":      frames,
        }

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 2 — Frame classification
    # ─────────────────────────────────────────────────────────────────────────

    def _is_copula_frame(self, frame: str) -> bool:
        """
        True if frame is a copula/auxiliary verb (meaningless on its own).
        These become edge labels instead of nodes.
        """
        tokens = frame.strip().lower()
        return bool(tokens) and tokens in self.PREDICATE_LEAD_WORDS

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 3 — Relation extraction (UPDATED)
    # ─────────────────────────────────────────────────────────────────────────

    def extract_relations(self, nodes):

      identities = nodes["identities"]
      professions = nodes["professions"]
      triggers = nodes["triggers"]
      frames = nodes["frames"]

      relations = []

      # ----------------------------
      # Identity ↔ Identity
      # ----------------------------
      if len(identities) > 1:
          relations.extend(
              self._build_intersections(identities)
          )

      # ----------------------------
      # Identity ↔ Profession
      # ----------------------------
      if professions:
          relations.extend(
              self._build_role_associations(
                  identities,
                  professions
              )
          )

      # ----------------------------
      # Frame-driven patterns
      # ----------------------------
      if len(frames) == 1 and len(triggers) > 1:
        frame_pairs = [(t, frames[0]) for t in triggers]

      else:
        frame_pairs = list(zip(triggers, frames))

      for trigger, frame in frame_pairs:

          if self._is_copula_frame(frame):

              relations.extend(
                  self._build_predicate_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

          else:

              relations.extend(
                  self._build_semantic_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

      return self._dedupe(relations)

    # ── Relation builders ─────────────────────────────────────────────────────

    def _build_intersections(
        self, identities: List[Dict[str, Any]]
    ) -> List[Dict[str, str]]:
        return [
            self._edge(a["span"], b["span"], self.REL_IDENTITY_INTERSECTION)
            for a, b in itertools.combinations(identities, 2)
        ]

    def _build_role_associations(
        self,
        identities: List[Dict[str, Any]],
        professions: List[str],
    ) -> List[Dict[str, str]]:
        return [
            self._edge(i["span"], p, self.REL_IDENTITY_ROLE_ASSOCIATION)
            for i in identities
            for p in professions
        ]

    def _build_predicate_relations(self,identities,professions,trigger,frame):

      relations = []

      for identity in identities:

          relations.append(
              self._edge(
                  identity["span"],
                  trigger,
                  frame
              )
          )

      for profession in professions:

          relations.append(
              self._edge(
                  profession,
                  trigger,
                  frame
              )
          )

      return relations

    def _build_semantic_relations(self,identities, professions,trigger,frame):

        relations = []

        # Identity → Frame

        for identity in identities:

            relations.append(
                self._edge(
                    identity["span"],
                    frame,
                    self.REL_IDENTITY_FRAME_ASSOCIATION
                )
            )

        # Profession → Frame

        for profession in professions:

            relations.append(
                self._edge(
                    profession,
                    frame,
                    self.REL_IDENTITY_ROLE_ASSOCIATION
                )
            )

        # Frame → Trigger

        relations.append(
            self._edge(
                frame,
                trigger,
                self.REL_CHARACTERIZED_AS
            )
        )

        return relations


    # ── Utilities ─────────────────────────────────────────────────────────────

    @staticmethod
    def _edge(source: str, target: str, relation: str) -> Dict[str, str]:
        return {"source": source, "target": target, "relation": relation}

    @staticmethod
    def _dedupe(rels: List[Dict[str, str]]) -> List[Dict[str, str]]:
        seen: set = set()
        unique: List[Dict[str, str]] = []
        for r in rels:
            k = (r["source"], r["target"], r["relation"])
            if k not in seen:
                seen.add(k)
                unique.append(r)
        return unique

    # ── Public entry points (unchanged) ───────────────────────────────────

    def process_annotation(self,annotation: Dict[str, Any]) -> List[Dict[str, str]]:
      nodes = self.extract_nodes(annotation)
      return self.extract_relations(nodes)

    def process_batch(self, annotations: List[Dict[str, Any]]) -> List[Dict[str, str]]:
        out: List[Dict[str, str]] = []
        for ann in annotations:
            out.extend(self.process_annotation(ann))
        return out

In [ ]:
# ── Per-sentence graph: typed nodes + safe handling of empty annotations ──────
# Each graph node is keyed by (normalised span, node type), so identical text
# assigned to different DER node types is preserved as distinct typed nodes.
# Empty annotations intentionally produce a sentence-only graph after add_sentence_node().

def _normalise_annotated_span(raw_span: str, node_type: str) -> str:
    raw = raw_span.strip()
    return _normalize_identity(raw) if node_type == "Identity Dimensions" else raw.lower().strip()


def _typed_node_id(span: str, node_type: str):
    return (span, node_type)


def _lookup_node_meta(span: str, raw_annotations: list) -> Tuple[str, Optional[str]]:
    """Compatibility helper: return the first matching annotation's type and label."""
    for ann in raw_annotations:
        raw = ann.get("span", "")
        if not isinstance(raw, str):
            continue
        node_type = ann.get("node", "Unknown")
        if _normalise_annotated_span(raw, node_type) == span:
            return node_type, ann.get("label")
    return "Unknown", None


def build_sentence_graph(
    annotation: Dict[str, Any],
    extractor: Optional[RuleBasedRelations] = None
) -> nx.DiGraph:
    if extractor is None:
        extractor = RuleBasedRelations()

    G = nx.DiGraph()
    raw_anns = annotation.get("annotations", []) or []

    # span -> all typed node IDs for that normalised text; keep duplicates by type.
    candidates = defaultdict(list)

    # 1. Add every valid annotation, including isolated nodes.
    for ann in raw_anns:
        raw_span = ann.get("span", "")
        if not isinstance(raw_span, str) or not raw_span.strip():
            continue
        node_type = ann.get("node", "Unknown")
        span = _normalise_annotated_span(raw_span, node_type)
        if not span:
            continue
        node_id = _typed_node_id(span, node_type)
        if node_id not in G:
            G.add_node(
                node_id,
                node_type=node_type,
                ontology_label=ann.get("label"),
                frequency=1,
                surface=raw_span.strip(),
            )
            candidates[span].append(node_id)
        else:
            G.nodes[node_id]["frequency"] += 1

    # Type expectations disambiguate relation endpoints that the current rule
    # extractor emits as plain text strings rather than typed node references.
    ID = RuleBasedRelations.IDENTITY_NODE
    PROF = RuleBasedRelations.PROFESSION_NODE
    TRIG = RuleBasedRelations.TRIGGER_NODE
    FRAME = RuleBasedRelations.FRAME_NODE
    expected_types = {
        RuleBasedRelations.REL_IDENTITY_INTERSECTION: ({ID}, {ID}),
        RuleBasedRelations.REL_CHARACTERIZED_AS: ({FRAME}, {TRIG}),
        RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION: ({ID}, {FRAME}),
        RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION: ({ID, PROF}, {PROF, FRAME}),
    }

    def resolve_endpoint(text, preferred_types=None):
        text = str(text).strip()
        matches = candidates.get(text, [])
        if preferred_types:
            preferred = [n for n in matches if G.nodes[n].get("node_type") in preferred_types]
            if preferred:
                matches = preferred
        if matches:
            return matches[0]
        # Preserve any unexpected relation endpoint as an Unknown typed node.
        node_id = _typed_node_id(text, "Unknown")
        if node_id not in G:
            G.add_node(node_id, node_type="Unknown", ontology_label=None,
                       frequency=1, surface=text)
        candidates[text].append(node_id)
        return node_id

    # 2. Add extracted relations, mapped onto type-aware node IDs.
    for rel in extractor.process_annotation(annotation):
        source_text, target_text = rel["source"], rel["target"]
        relation = rel["relation"]
        src_types, tgt_types = expected_types.get(relation, ({ID, PROF}, {TRIG}))
        source = resolve_endpoint(source_text, src_types)
        target = resolve_endpoint(target_text, tgt_types)
        if G.has_edge(source, target):
            G[source][target]["weight"] += 1
        else:
            G.add_edge(source, target, relation=relation, weight=1)

    G.graph["sentence"] = annotation.get("sentence", "")
    return G


In [ ]:
SENTENCE_NODE_TYPE = "Sentence"
SENTENCE_RELATION = "contains"


def add_sentence_node(G):
    """
    Add a dedicated Sentence node to an existing
    sentence-level NetworkX graph.
    """

    G = G.copy()

    sentence = G.graph.get("sentence", "")

    # Unique node name for this graph
    sentence_node = "__SENTENCE__"

    # Add sentence node
    G.add_node(
        sentence_node,
        node_type=SENTENCE_NODE_TYPE,
        ontology_label=None,
        frequency=1,
        text=sentence
    )

    # Connect Sentence -> every existing semantic node
    for node in list(G.nodes()):

        if node == sentence_node:
            continue

        G.add_edge(
            sentence_node,
            node,
            relation=SENTENCE_RELATION,
            weight=1
        )

    return G

In [ ]:
# Remember each graph node's original surface span (needed for contextual BERT alignment).
IDENTITY_NODE_NAME = "Identity Dimensions"

def _raw_span_for_graph_node(node, node_attrs, annotation):
    """Return the stored original span; fall back to annotation matching for older graphs."""
    if node_attrs.get("surface"):
        return str(node_attrs["surface"])
    ntype = node_attrs.get("node_type", "Unknown")
    node_text = node[0] if isinstance(node, tuple) and len(node) == 2 else str(node)
    for ann in annotation.get("annotations", []):
        if ann.get("node", "Unknown") != ntype:
            continue
        raw = str(ann.get("span", "")).strip()
        if not raw:
            continue
        cand = _normalize_identity(raw) if ntype == IDENTITY_NODE_NAME else raw.lower().strip()
        if cand == str(node_text).lower().strip():
            return raw
    return str(node_text)


def attach_surfaces(G, annotation):
    for n, a in G.nodes(data=True):
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            continue
        a["surface"] = _raw_span_for_graph_node(n, a, annotation)
    return G


In [ ]:
IDENTITY_NODE = RuleBasedRelations.IDENTITY_NODE
CANONICAL_RELATIONS = {
    RuleBasedRelations.REL_IDENTITY_INTERSECTION,
    RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION,
    RuleBasedRelations.REL_CHARACTERIZED_AS,
    RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION,
    SENTENCE_RELATION,
}


def relation_family(rel: str) -> str:
    """Collapse the open-ended relation vocabulary (copula words such as 'is', 'can', ...)
    into 'predicate'; canonical DER relations and 'contains' are kept. Strips 'rev_'."""
    r = rel[4:] if rel.startswith("rev_") else rel
    return r if r in CANONICAL_RELATIONS else "predicate"


# ── NetworkX-level graph edits (used by ablations / counterfactuals / occlusion) ──
def nx_remove_der_edges(G, families=None):
    """Remove DER edges (everything except Sentence->node 'contains').
    families=None removes all; otherwise only edges whose relation_family is in `families`."""
    G2 = G.copy()
    drop = [(u, v) for u, v, a in G2.edges(data=True)
            if a["relation"] != SENTENCE_RELATION
            and (families is None or relation_family(a["relation"]) in families)]
    G2.remove_edges_from(drop)
    return G2


def nx_remove_node_type(G, node_type):
    G2 = G.copy()
    G2.remove_nodes_from([n for n, a in G2.nodes(data=True) if a.get("node_type") == node_type])
    return G2


# ── Build graphs for each dataset ───────────────────────────────────────────
extractor = RuleBasedRelations()


def build_dataset(annos_pos, annos_neg):
    annos = list(annos_pos) + list(annos_neg)
    labels = np.array([1] * len(annos_pos) + [0] * len(annos_neg))
    graphs = [attach_surfaces(add_sentence_node(build_sentence_graph(a, extractor)), a) for a in annos]
    # Groups: same template_id => same group (pair-safe CV). Items without one get their own group.
    groups = np.array([a["template_id"] if a.get("template_id") is not None else -(i + 1)
                       for i, a in enumerate(annos)])
    return dict(annos=annos, graphs=graphs, labels=labels, groups=groups)


DATA = {name: build_dataset(d["pos"], d["neg"]) for name, d in DATASETS.items()}
for name, d in DATA.items():
    n_nodes = np.mean([G.number_of_nodes() - 1 for G in d["graphs"]])
    n_der = np.mean([sum(a["relation"] != SENTENCE_RELATION for _, _, a in G.edges(data=True))
                     for G in d["graphs"]])
    print(f"{name}: {len(d['graphs'])} graphs | avg entity nodes {n_nodes:.1f} | avg DER edges {n_der:.1f}"
          f" | groups {len(set(d['groups']))}")

InDiBias: 326 graphs | avg entity nodes 2.8 | avg DER edges 1.6 | groups 326
IndiCasa: 326 graphs | avg entity nodes 5.8 | avg DER edges 4.7 | groups 326


In [ ]:
# ── Graph coverage audit (run after DATA is built) ────────────────────────────
# Empty annotations are expected to have zero DER/entity nodes; they are not
# silently repaired or assigned invented relationships.
for name, d in DATA.items():
    rows = []
    for i, (ann, G, y) in enumerate(zip(d["annos"], d["graphs"], d["labels"])):
        entity_nodes = [(n, a) for n, a in G.nodes(data=True)
                        if a.get("node_type") != SENTENCE_NODE_TYPE]
        der_edges = [(u, v, a) for u, v, a in G.edges(data=True)
                     if a.get("relation") != SENTENCE_RELATION]
        if not ann.get("annotations") or not entity_nodes:
            rows.append({
                "dataset": name, "index": i, "label": int(y),
                "sentence": ann.get("sentence", ""),
                "annotation_count": len(ann.get("annotations", []) or []),
                "entity_node_count": len(entity_nodes),
                "der_edge_count": len(der_edges),
                "empty_annotations": not bool(ann.get("annotations")),
            })
    print(f"{name}: {len(rows)} records with empty annotations or no entity nodes")
    if rows:
        display(pd.DataFrame(rows).head(10))


InDiBias: 40 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,InDiBias,27,1,They were asked to be careful while walking th...,0,0,0,True
1,InDiBias,73,1,Prakash learned to play chess very quickly and...,0,0,0,True
2,InDiBias,74,1,Anjali was terrified of bugs.,0,0,0,True
3,InDiBias,78,1,Kabir will not let his sister go for buying mi...,0,0,0,True
4,InDiBias,81,1,Saira served food in a restaurant.,0,0,0,True
5,InDiBias,86,1,Imran was the fastest on the draw and shot roh...,0,0,0,True
6,InDiBias,87,1,Raj took off his ring and put it in his pocket...,0,0,0,True
7,InDiBias,88,1,Raj took his wedding ring off so it wouldn't g...,0,0,0,True
8,InDiBias,89,1,Sunil spent most of his time writing programs ...,0,0,0,True
9,InDiBias,96,1,i'm glad i wasn't there since i would have bee...,0,0,0,True


IndiCasa: 1 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,IndiCasa,108,1,The brother will not let his sister go for buy...,0,0,0,True


In [ ]:
import hashlib
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

def make_splits(labels, groups, strategy, seeds, n_splits=5):
    reps = []
    for s in seeds:
        if strategy == "stratified":
            it = StratifiedKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels)
        elif strategy == "grouped":
            it = StratifiedGroupKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels, groups)
        else:
            raise ValueError(strategy)
        reps.append([(tr, te) for tr, te in it])
    return reps

name = "InDiBias"
y = np.asarray(DATA[name]["labels"])
groups = np.asarray(DATA[name]["groups"])

print("Number of examples:", len(y))
print("Labels hash:", hashlib.sha256(y.tobytes()).hexdigest())
print("Groups hash:", hashlib.sha256(groups.tobytes()).hexdigest())
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Unique groups:", len(np.unique(groups)))

# "grouped" dropped from the eval plan (no template_id in the annotation files -- see README);
splits = make_splits(y, groups, "stratified", [42, 142, 242])

fold_hashes = []
for repeat, folds in enumerate(splits):
    for fold, (train_idx, test_idx) in enumerate(folds):
        # Check there is no group overlap
        assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))

        # Hash the exact train/test indices
        h = hashlib.sha256(
            np.concatenate([train_idx, [-1], test_idx]).tobytes()
        ).hexdigest()
        fold_hashes.append(h)

print("Fold hashes:")
print(fold_hashes)

## 3. Node-feature encoders (frozen)

| Encoder | DER-node vector | Sentence-node vector | What it tests |
|---|---|---|---|
| `SBERT` | `all-MiniLM-L6-v2` embedding of node text in isolation (384d) | full-sentence embedding | strong compact sentence/node baseline |
| `BERT` | `bert-base-uncased` span tokens mean-pooled **in sentence context** (768d) | `[CLS]` for full sentence | contextual span representation |
| `BERT-ISO` | `bert-base-uncased` `[CLS]` for the node text **in isolation** (768d) | `[CLS]` for full sentence | tests whether sentence context blurs identity/lexical node signal |

All encoders are frozen. The R-GCN's per-node-type input projections already provide type-specific transformations; the representation ablations change only the text vectors. The no-sentence-feature ablation zeroes the Sentence node while retaining the DER nodes and edges.

**Alignment note:** contextual BERT requires matching annotation spans to character offsets. Isolated BERT does not, so it avoids contextual span-alignment failures for DER nodes. Alignment diagnostics are still reported for contextual BERT.

In [ ]:
# ── Build FEATURIZERS fresh (this notebook previously assumed FEATURIZERS already existed in
# the kernel from a prior notebook run in the same Colab session -- it does not define its own
# featurizer classes anywhere, so running this notebook standalone would NameError as soon as any
# later cell touches FEATURIZERS["SBERT"]. Fixed: build everything fresh via the shared module,
# so this notebook is now independently runnable, matching the RGAT notebook. ──────────────────
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
from src.data.featurizers import build_featurizers, sentence_node_name, node_surface, node_ctx, find_char_span

FEATURIZERS = build_featurizers(device, DATA, encoder_fp16=ENCODER_FP16, enc_batch=ENC_BATCH,
                                which=("SBERT", "BERT-ISO", "BERT-CTX"))


In [ ]:
def nx_to_hetero(G, label, feat, collapse_relations=False, zero_sentence=False):
    """Convert a NetworkX sentence graph to a DER-only HeteroData graph.

    The Sentence node is deliberately excluded. Only annotated DER nodes,
    their text-only features, and relations between DER nodes reach the R-GCN.
    Edges touching the Sentence node are discarded.
    """
    data = HeteroData()
    node_maps = {}
    der_nodes = [(n, a) for n, a in G.nodes(data=True)
                 if a.get("node_type", "Unknown") != SENTENCE_NODE_TYPE]
    node_types = sorted({a.get("node_type", "Unknown") for _, a in der_nodes})

    for nt in node_types:
        nodes = [n for n, a in der_nodes if a.get("node_type", "Unknown") == nt]
        node_maps[nt] = {n: i for i, n in enumerate(nodes)}
        feat.ensure([feat.node_key(G, n) for n in nodes])
        data[nt].x = torch.stack([feat.get(G, n) for n in nodes]).float()

    edge_dict = defaultdict(list)
    for s, t, a in G.edges(data=True):
        st = G.nodes[s].get("node_type", "Unknown")
        tt = G.nodes[t].get("node_type", "Unknown")
        # No Sentence node or Sentence-incident edges are allowed into the encoder.
        if st == SENTENCE_NODE_TYPE or tt == SENTENCE_NODE_TYPE:
            continue
        if st not in node_maps or tt not in node_maps:
            continue
        rel = a.get("relation", "unknown")
        if collapse_relations:
            rel = relation_family(rel)
        si, ti = node_maps[st][s], node_maps[tt][t]
        edge_dict[(st, rel, tt)].append([si, ti])
        edge_dict[(tt, "rev_" + rel, st)].append([ti, si])

    for et, e in edge_dict.items():
        data[et].edge_index = torch.tensor(e, dtype=torch.long).t().contiguous()
    data.y = torch.tensor([int(label)], dtype=torch.long)
    return data


def convert(graphs_nx, labels, feat, **kw):
    return [nx_to_hetero(G, y, feat, **kw) for G, y in zip(graphs_nx, labels)]


def build_metadata(data_list):
    nts, ets = set(), set()
    for g in data_list:
        n, e = g.metadata()
        nts.update(n); ets.update(e)
    return sorted(nts), sorted(ets)


In [ ]:
class SentenceRGCN(nn.Module):
    """DER-only R-GCN: typed DER-node projections -> 2x RGCNConv -> graph mean pool -> classify.

    There is no Sentence node/readout. Only DER node features and DER-to-DER
    relations are inputs. Graphs with no DER nodes receive a zero pooled vector.
    """
    def __init__(self, metadata, input_channels=384, hidden_channels=64,
                 num_classes=2, dropout=0.3, num_bases=None):
        super().__init__()
        node_types, edge_types = metadata
        self.node_types, self.edge_types = list(node_types), list(edge_types)
        if SENTENCE_NODE_TYPE in self.node_types:
            raise ValueError("Sentence node must not be present in DER-only R-GCN metadata")
        self.edge_type_to_id = {et: i for i, et in enumerate(self.edge_types)}
        self.input_proj = nn.ModuleDict({nt: nn.Linear(input_channels, hidden_channels)
                                         for nt in self.node_types})
        R = max(1, len(self.edge_types))
        self.rgcn1 = RGCNConv(hidden_channels, hidden_channels, num_relations=R, num_bases=num_bases)
        self.rgcn2 = RGCNConv(hidden_channels, hidden_channels, num_relations=R, num_bases=num_bases)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, data):
        n_graphs = int(getattr(data, "num_graphs", 1))
        x_parts, batch_parts, offsets, offset = [], [], {}, 0
        for nt in self.node_types:
            if nt not in data.node_types:
                continue
            node_x = data[nt].x
            h = self.input_proj[nt](node_x)
            x_parts.append(h)
            offsets[nt] = offset
            offset += h.size(0)
            if hasattr(data[nt], "batch") and data[nt].batch is not None:
                batch_parts.append(data[nt].batch.to(h.device))
            else:
                batch_parts.append(torch.zeros(h.size(0), dtype=torch.long, device=h.device))

        if not x_parts:
            pooled = torch.zeros((n_graphs, self.classifier[0].in_features),
                                 device=next(self.parameters()).device)
            return self.classifier(pooled)

        x = torch.cat(x_parts, dim=0)
        node_batch = torch.cat(batch_parts, dim=0)
        if x.size(0) == 0:
            pooled = x.new_zeros((n_graphs, self.classifier[0].in_features))
            return self.classifier(pooled)

        edges, rel_ids = [], []
        for et in data.edge_types:
            src_t, _, dst_t = et
            if et not in self.edge_type_to_id or src_t not in offsets or dst_t not in offsets:
                continue
            ei = data[et].edge_index
            if ei.numel() == 0:
                continue
            ei = ei.clone()
            ei[0] += offsets[src_t]
            ei[1] += offsets[dst_t]
            edges.append(ei)
            rel_ids.append(torch.full((ei.size(1),), self.edge_type_to_id[et],
                                      dtype=torch.long, device=x.device))

        if edges:
            edge_index = torch.cat(edges, dim=1)
            edge_type = torch.cat(rel_ids)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long, device=x.device)
            edge_type = torch.empty(0, dtype=torch.long, device=x.device)

        x = F.relu(self.rgcn1(x, edge_index, edge_type))
        x = F.relu(self.rgcn2(x, edge_index, edge_type))

        # Mean-pool all DER nodes belonging to each graph; no sentence embedding is used.
        pooled = x.new_zeros((n_graphs, x.size(-1)))
        pooled.index_add_(0, node_batch, x)
        counts = x.new_zeros((n_graphs, 1))
        counts.index_add_(0, node_batch, x.new_ones((x.size(0), 1)))
        pooled = pooled / counts.clamp_min(1.0)
        return self.classifier(pooled)


## R-GCN-only evaluation harness

All nine R-GCN variants use the same folds and seeds. Baselines are deliberately excluded from the training loop. `raw` keeps original relation names; `collapsed` maps relations through the notebook's existing `relation_family()` function.


In [ ]:
def make_splits(labels, groups, strategy, seeds, n_splits=5):
    reps = []
    for s in seeds:
        if strategy == "stratified":
            it = StratifiedKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels)
        elif strategy == "grouped":
            it = StratifiedGroupKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels, groups)
        else:
            raise ValueError(strategy)
        reps.append([(tr, te) for tr, te in it])
    return reps


def cross_validate(fit_predict, labels, reps, seeds):
    """Out-of-fold probabilities [n_repeats, N, 2]."""
    oof = np.zeros((len(reps), len(labels), 2))
    for r, (folds, s) in enumerate(zip(reps, seeds)):
        for f, (tr, te) in enumerate(folds, start=1):
            oof[r, te] = fit_predict(tr, te, s + f)
    return oof


def summarize(oof, labels, name):
    rows = []
    for r in range(oof.shape[0]):
        p = np.clip(oof[r], 1e-7, 1 - 1e-7)
        p = p / p.sum(1, keepdims=True)
        pred = p.argmax(1)
        rows.append((accuracy_score(labels, pred), f1_score(labels, pred, average="macro"),
                     log_loss(labels, p, labels=[0, 1])))
    a = np.array(rows)
    return {"method": name, "acc": a[:, 0].mean(), "acc_sd": a[:, 0].std(), "macro_f1": a[:, 1].mean(),
            "log_loss": a[:, 2].mean(), "n_repeats": oof.shape[0]}


def paired_bootstrap(oof_a, oof_b, labels, groups, n_boot=5000, seed=0):
    """Does A beat B? Bootstrap over *groups* (templates) so minimal pairs are resampled together."""
    ca = (oof_a.argmax(2) == labels).mean(0)
    cb = (oof_b.argmax(2) == labels).mean(0)
    diff = ca - cb
    ug = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in ug}
    rng = np.random.default_rng(seed)
    boots = np.array([np.concatenate([diff[idx_by[g]] for g in rng.choice(ug, len(ug), replace=True)]).mean()
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    return dict(acc_diff=diff.mean(), ci_low=lo, ci_high=hi, p_two_sided=p)


def holm(pvals):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj, run = np.empty(m), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj


In [ ]:
# ═════════ R-GCN-only registry and training helpers ═════════
from dataclasses import dataclass

@dataclass(frozen=True)
class RGCNVariant:
    name: str
    feats: str           # SBERT or BERT-ISO
    cfg_tag: str         # base or reg
    collapse: bool
    @property
    def cfg(self):
        return {"base": BASE_CFG, "reg": REG_CFG}[self.cfg_tag]

VARIANTS = {}
for _feats, _label in [("SBERT", "SBERT"), ("BERT-ISO", "BERT-iso")]:
    for _collapse, _tag, _rel in [(False, "base", "raw"), (True, "base", "collapsed"), (True, "reg", "collapsed")]:
        _v = RGCNVariant(f"R-GCN {_label} {_rel} ({_tag})", _feats, _tag, _collapse)
        VARIANTS[_v.name] = _v

print(f"Rerunning {len(VARIANTS)} R-GCN variants only:")
for _name in VARIANTS: print(" -", _name)

# FEATURIZERS is already built (SBERT, BERT-ISO, BERT-CTX) by the cell under "## 3. Node-feature encoders" above -- this cell used to rebuild it from scratch with only SBERT/BERT-ISO, which silently dropped BERT-CTX and would KeyError as soon as the GAT section below tried to use it. Do not rebuild FEATURIZERS here.

# Cache graph tensors by dataset/encoder/relation setting.
HCACHE = {}
def get_hetero(name, feats, collapse):
    key = (name, feats, collapse)
    if key not in HCACHE:
        HCACHE[key] = convert(DATA[name]["graphs"], DATA[name]["labels"], FEATURIZERS[feats],
                              collapse_relations=collapse)
        # Hard guard: no sentence node type is allowed in any converted graph.
        for _g in HCACHE[key]:
            if SENTENCE_NODE_TYPE in _g.node_types:
                raise AssertionError("Sentence node leaked into DER-only R-GCN input")
    return HCACHE[key]


def fit_rgcn(train_graphs, metadata, cfg, seed, input_dim):
    set_seed(seed)
    loader = DataLoader(train_graphs, batch_size=cfg["batch_size"], shuffle=True)
    net = SentenceRGCN(metadata, input_channels=input_dim, hidden_channels=cfg["hidden"],
                       dropout=cfg["dropout"], num_bases=cfg["num_bases"]).to(train_device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    for _ in range(cfg["epochs"]):
        net.train()
        for batch in loader:
            batch = batch.to(train_device)
            opt.zero_grad()
            F.cross_entropy(net(batch), batch.y).backward()
            opt.step()
    return net

@torch.no_grad()
def predict_proba_net(net, graphs, batch_size=64):
    net.eval()
    dev = next(net.parameters()).device
    outs = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        outs.append(F.softmax(net(batch.to(dev)), dim=1).cpu())
    return torch.cat(outs).numpy()

# Prebuild fold-independent tensor graphs. Only R-GCN variants are constructed.


## Run the R-GCN variants

This is the only training cell in this notebook. It retrains the nine R-GCN variants and writes a separate CSV. The previously trained non-R-GCN models are not touched.


In [ ]:
# Fix: provide the missing get() method for all Featurizer subclasses
def _featurizer_get(self, G, n):
    return self.cache[self.node_key(G, n)]

Featurizer.get = _featurizer_get

# Quick sanity check
for name, feat in FEATURIZERS.items():
    print(name, "has get():", callable(getattr(feat, "get", None)))

BERT-ISO has get(): False
SBERT has get(): False


In [ ]:
# ── Patch featurizer instances to guarantee get() exists ───────────────────
for name, feat in FEATURIZERS.items():
    if not hasattr(feat, "get"):
        def _make_get(f):
            return lambda G, n: f.cache.get(f.node_key(G, n))
        feat.get = _make_get(feat)

RES, SPL = {}, {}
t_all = time.time()
for plan in EVAL_PLAN:
    name, cv = plan
    d = DATA[name]; y = d["labels"]
    SPL[plan] = make_splits(y, d["groups"], cv, SEEDS)
    RES[plan] = {}
    for vname, v in VARIANTS.items():
        t0 = time.time()
        data_list = get_hetero(name, v.feats, v.collapse)
        metadata = build_metadata(data_list)
        def _fit_predict(tr, te, seed, _data=data_list, _metadata=metadata, _v=v):
            net = fit_rgcn([_data[i] for i in tr], _metadata, _v.cfg, seed, FEATURIZERS[_v.feats].dim)
            probs = predict_proba_net(net, [_data[i] for i in te])
            del net
            free_gpu()
            return probs
        RES[plan][vname] = cross_validate(_fit_predict, y, SPL[plan], SEEDS)
        summary = summarize(RES[plan][vname], y, vname)
        print(f"{name:9s}/{cv:10s} {vname:32s} acc={summary['acc']:.3f} "
              f"macro-F1={summary['macro_f1']:.3f} log-loss={summary['log_loss']:.3f} "
              f"({time.time()-t0:.0f}s)")
    free_gpu(); gpu_report(f"after {name}/{cv}")
print(f"R-GCN-only evaluation completed in {(time.time()-t_all)/60:.1f} min")

InDiBias /grouped    R-GCN SBERT raw (base)           acc=0.745 macro-F1=0.431 log-loss=0.563 (269s)
InDiBias /grouped    R-GCN SBERT collapsed (base)     acc=0.745 macro-F1=0.431 log-loss=0.563 (176s)
InDiBias /grouped    R-GCN SBERT collapsed (reg)      acc=0.745 macro-F1=0.427 log-loss=0.554 (102s)
InDiBias /grouped    R-GCN BERT-iso raw (base)        acc=0.753 macro-F1=0.456 log-loss=0.551 (180s)
InDiBias /grouped    R-GCN BERT-iso collapsed (base)  acc=0.753 macro-F1=0.456 log-loss=0.551 (181s)
InDiBias /grouped    R-GCN BERT-iso collapsed (reg)   acc=0.745 macro-F1=0.427 log-loss=0.554 (104s)
[gpu] after InDiBias/grouped: running on CPU
InDiBias /stratified R-GCN SBERT raw (base)           acc=0.740 macro-F1=0.436 log-loss=0.560 (175s)
InDiBias /stratified R-GCN SBERT collapsed (base)     acc=0.740 macro-F1=0.436 log-loss=0.560 (175s)
InDiBias /stratified R-GCN SBERT collapsed (reg)      acc=0.745 macro-F1=0.427 log-loss=0.546 (101s)
InDiBias /stratified R-GCN BERT-iso raw (base)

In [ ]:
rows = []
for plan, methods in RES.items():
    y = DATA[plan[0]]["labels"]
    for model_name, oof in methods.items():
        r = summarize(oof, y, model_name)
        v = VARIANTS[model_name]
        r.update(dataset=plan[0], cv=plan[1], encoder=v.feats,
                 relations="collapsed" if v.collapse else "raw", config=v.cfg_tag)
        rows.append(r)
rgcn_results_df = pd.DataFrame(rows)[
    ["dataset", "cv", "method", "encoder", "relations", "config", "acc", "acc_sd", "macro_f1", "log_loss", "n_repeats"]
]
display(rgcn_results_df.round(4))
rgcn_results_df.to_csv("results/rgcn_only_results.csv", index=False)
print("Saved: results/rgcn_only_results.csv")


,dataset,cv,method,encoder,relations,config,acc,acc_sd,macro_f1,log_loss,n_repeats
0,InDiBias,grouped,R-GCN SBERT raw (base),SBERT,raw,base,0.7454,0.0025,0.4309,0.5626,3
1,InDiBias,grouped,R-GCN SBERT collapsed (base),SBERT,collapsed,base,0.7454,0.0025,0.4309,0.5626,3
2,InDiBias,grouped,R-GCN SBERT collapsed (reg),SBERT,collapsed,reg,0.7454,0.0000,0.4271,0.5540,3
3,InDiBias,grouped,R-GCN BERT-iso raw (base),BERT-ISO,raw,base,0.7526,0.0038,0.4560,0.5513,3
4,InDiBias,grouped,R-GCN BERT-iso collapsed (base),BERT-ISO,collapsed,base,0.7526,0.0038,0.4560,0.5513,3
5,InDiBias,grouped,R-GCN BERT-iso collapsed (reg),BERT-ISO,collapsed,reg,0.7454,0.0000,0.4271,0.5542,3
6,InDiBias,stratified,R-GCN SBERT raw (base),SBERT,raw,base,0.7403,0.0029,0.4362,0.5597,3
7,InDiBias,stratified,R-GCN SBERT collapsed (base),SBERT,collapsed,base,0.7403,0.0029,0.4362,0.5597,3
8,InDiBias,stratified,R-GCN SBERT collapsed (reg),SBERT,collapsed,reg,0.7454,0.0000,0.4271,0.5464,3
9,InDiBias,stratified,R-GCN BERT-iso raw (base),BERT-ISO,raw,base,0.7464,0.0081,0.4499,0.5699,3


Saved: results/rgcn_only_results.csv


## GAT comparison — matched nine-variant evaluation

This section adds a **relation-aware Graph Attention Network (GAT)** baseline without modifying or replacing any R-GCN code or outputs. It evaluates the same nine encoder/relation/configuration combinations: SBERT (3), BERT-iso (3), and BERT-ctx (3).

The GAT uses the same DER-only `HeteroData` graphs, node-span features, `BASE_CFG` / `REG_CFG`, evaluation plans, folds, seeds, and metrics as the R-GCN experiments. Sentence nodes are prohibited. Unlike vanilla GAT, this implementation supplies a learned edge-relation embedding to attention, so raw versus collapsed relation settings remain distinguishable. Results are written separately to `results/gat_nine_variants_results.csv` (and a matched comparison file when the R-GCN results are still in memory).

Run this section **after the existing BERT-ctx feature-cache cell**. It does not rerun, overwrite, or change the R-GCN results.

In [ ]:
# Relation-aware GAT: same graph-level task and training/evaluation protocol as the R-GCN.
# Edge relation IDs are embedded and supplied to GATConv as edge attributes.
from torch_geometric.nn import GATConv

class DEROnlyGAT(nn.Module):
    """Two-layer relation-aware GAT over DER nodes only; mean-pool node states per graph."""
    def __init__(self, metadata, input_channels=384, hidden_channels=64,
                 num_classes=2, dropout=0.3, heads=4, relation_dim=32):
        super().__init__()
        node_types, edge_types = metadata
        self.node_types, self.edge_types = list(node_types), list(edge_types)
        if SENTENCE_NODE_TYPE in self.node_types:
            raise ValueError("Sentence node must not be present in DER-only GAT metadata")
        self.edge_type_to_id = {et: i for i, et in enumerate(self.edge_types)}
        self.input_proj = nn.ModuleDict({nt: nn.Linear(input_channels, hidden_channels)
                                         for nt in self.node_types})
        self.num_relations = max(1, len(self.edge_types))
        self.rel_embedding = nn.Embedding(self.num_relations, relation_dim)
        self.gat1 = GATConv(hidden_channels, hidden_channels, heads=heads, concat=False,
                            dropout=dropout, edge_dim=relation_dim, add_self_loops=False)
        self.gat2 = GATConv(hidden_channels, hidden_channels, heads=heads, concat=False,
                            dropout=dropout, edge_dim=relation_dim, add_self_loops=False)
        self.skip1 = nn.Linear(hidden_channels, hidden_channels)
        self.skip2 = nn.Linear(hidden_channels, hidden_channels)
        self.dropout = float(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

    def forward(self, data):
        n_graphs = int(getattr(data, "num_graphs", 1))
        x_parts, batch_parts, offsets, offset = [], [], {}, 0
        for nt in self.node_types:
            if nt not in data.node_types:
                continue
            node_x = data[nt].x
            h = self.input_proj[nt](node_x)
            x_parts.append(h)
            offsets[nt] = offset
            offset += h.size(0)
            batch_vec = getattr(data[nt], "batch", None)
            if batch_vec is not None:
                batch_parts.append(batch_vec.to(h.device))
            else:
                batch_parts.append(torch.zeros(h.size(0), dtype=torch.long, device=h.device))

        if not x_parts:
            pooled = torch.zeros((n_graphs, self.classifier[0].in_features),
                                 device=next(self.parameters()).device)
            return self.classifier(pooled)

        x = torch.cat(x_parts, dim=0)
        node_batch = torch.cat(batch_parts, dim=0)
        if x.size(0) == 0:
            pooled = x.new_zeros((n_graphs, self.classifier[0].in_features))
            return self.classifier(pooled)

        edge_parts, rel_parts = [], []
        for et in data.edge_types:
            src_t, _, dst_t = et
            if et not in self.edge_type_to_id or src_t not in offsets or dst_t not in offsets:
                continue
            ei = data[et].edge_index
            if ei.numel() == 0:
                continue
            ei = ei.clone()
            ei[0] += offsets[src_t]
            ei[1] += offsets[dst_t]
            edge_parts.append(ei)
            rel_parts.append(torch.full((ei.size(1),), self.edge_type_to_id[et],
                                        dtype=torch.long, device=x.device))

        if edge_parts:
            edge_index = torch.cat(edge_parts, dim=1)
            relation_ids = torch.cat(rel_parts)
            edge_attr = self.rel_embedding(relation_ids)
            h = F.elu(self.gat1(x, edge_index, edge_attr=edge_attr) + self.skip1(x))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = F.elu(self.gat2(h, edge_index, edge_attr=edge_attr) + self.skip2(h))
        else:
            # Valid fallback for a graph/batch with no DER-to-DER edges.
            h = F.elu(self.skip1(x))
            h = F.elu(self.skip2(h))

        pooled = h.new_zeros((n_graphs, h.size(-1)))
        pooled.index_add_(0, node_batch, h)
        counts = h.new_zeros((n_graphs, 1))
        counts.index_add_(0, node_batch, h.new_ones((h.size(0), 1)))
        pooled = pooled / counts.clamp_min(1.0)
        return self.classifier(pooled)


def fit_gat(train_graphs, metadata, cfg, seed, input_dim):
    set_seed(seed)
    loader = DataLoader(train_graphs, batch_size=cfg["batch_size"], shuffle=True)
    net = DEROnlyGAT(metadata, input_channels=input_dim,
                     hidden_channels=cfg["hidden"], dropout=cfg["dropout"]).to(train_device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    for _ in range(cfg["epochs"]):
        net.train()
        for batch in loader:
            batch = batch.to(train_device)
            opt.zero_grad()
            F.cross_entropy(net(batch), batch.y).backward()
            opt.step()
    return net

print("DEROnlyGAT defined. No R-GCN definitions or results were changed.")


DEROnlyGAT defined. No R-GCN definitions or results were changed.


### Run all nine GAT variants

This mirrors the R-GCN main-results loop across all three evaluation plans. The three BERT-ctx variants are included here rather than requiring a separate manual run.

In [ ]:
from dataclasses import dataclass as _dataclass

@_dataclass(frozen=True)
class GATVariant:
    name: str
    feats: str
    cfg_tag: str
    collapse: bool
    @property
    def cfg(self):
        return {"base": BASE_CFG, "reg": REG_CFG}[self.cfg_tag]

GAT_VARIANTS = {}
for _feats, _label in [("SBERT", "SBERT"), ("BERT-ISO", "BERT-iso"), ("BERT-CTX", "BERT-ctx")]:
    for _collapse, _rel, _cfg in [(False, "raw", "base"), (True, "collapsed", "base"), (True, "collapsed", "reg")]:
        _v = GATVariant(f"GAT {_label} {_rel} ({_cfg})", _feats, _cfg, _collapse)
        GAT_VARIANTS[_v.name] = _v

assert len(GAT_VARIANTS) == 9, f"Expected nine GAT variants, got {len(GAT_VARIANTS)}"
print("Running these nine matched GAT variants:")
for _name in GAT_VARIANTS:
    print(" -", _name)

# Cache converted DER-only graphs. Reuse existing caches for SBERT/BERT-iso;
# create contextual graphs using the same converter and BERT-ctx span cache.
GAT_GRAPH_CACHE = {}
def get_gat_graphs(dataset_name, variant):
    key = (dataset_name, variant.feats, variant.collapse)
    if key not in GAT_GRAPH_CACHE:
        if variant.feats in ("SBERT", "BERT-ISO"):
            graphs = get_hetero(dataset_name, variant.feats, variant.collapse)
        else:
            graphs = convert(DATA[dataset_name]["graphs"], DATA[dataset_name]["labels"],
                             FEATURIZERS["BERT-CTX"], collapse_relations=variant.collapse)
        for graph in graphs:
            if SENTENCE_NODE_TYPE in graph.node_types:
                raise AssertionError("Sentence node leaked into DER-only GAT input")
        GAT_GRAPH_CACHE[key] = graphs
    return GAT_GRAPH_CACHE[key]

GAT_RES, GAT_SPL = {}, {}
gat_rows = []
t_gat_all = time.time()
for plan in EVAL_PLAN:
    dataset_name, cv_name = plan
    y = DATA[dataset_name]["labels"]
    groups = DATA[dataset_name]["groups"]
    if plan not in SPL:
        SPL[plan] = make_splits(y, groups, cv_name, SEEDS)
    GAT_SPL[plan] = SPL[plan]  # exact same fold indices and repeat seeds as R-GCN
    GAT_RES[plan] = {}

    for variant_name, variant in GAT_VARIANTS.items():
        t0 = time.time()
        graph_list = get_gat_graphs(dataset_name, variant)
        metadata = build_metadata(graph_list)
        input_dim = FEATURIZERS[variant.feats].dim

        def _fit_predict_gat(tr, te, seed, _graphs=graph_list, _meta=metadata, _v=variant, _dim=input_dim):
            net = fit_gat([_graphs[i] for i in tr], _meta, _v.cfg, seed, _dim)
            probs = predict_proba_net(net, [_graphs[i] for i in te])
            del net
            free_gpu()
            return probs

        oof = cross_validate(_fit_predict_gat, y, GAT_SPL[plan], SEEDS)
        GAT_RES[plan][variant_name] = oof
        row = summarize(oof, y, variant_name)
        row.update(dataset=dataset_name, cv=cv_name, model="GAT", encoder=variant.feats,
                   relations="collapsed" if variant.collapse else "raw", config=variant.cfg_tag)
        gat_rows.append(row)
        print(f"{dataset_name:9s}/{cv_name:10s} {variant_name:32s} "
              f"acc={row['acc']:.3f} macro-F1={row['macro_f1']:.3f} "
              f"log-loss={row['log_loss']:.3f} ({time.time()-t0:.0f}s)")
    free_gpu()
    gpu_report(f"after GAT {dataset_name}/{cv_name}")

gat_df = pd.DataFrame(gat_rows)
gat_path = "results/gat_nine_variants_results.csv"
gat_df.to_csv(gat_path, index=False)
print(f"\nGAT evaluation completed in {(time.time()-t_gat_all)/60:.1f} min")
print("Saved:", gat_path)
display(gat_df[["dataset", "cv", "variant", "acc", "acc_sd", "macro_f1", "log_loss"]].round(4))

# Optional matched comparison CSV, using R-GCN OOF arrays already in memory.
# This is only a report of both sets of results; it does not alter either model's scores.
comparison_rows = []
for plan in EVAL_PLAN:
    dataset_name, cv_name = plan
    y = DATA[dataset_name]["labels"]
    if "RES" in globals():
        for vname, oof in RES.get(plan, {}).items():
            rr = summarize(oof, y, vname)
            rr.update(dataset=dataset_name, cv=cv_name, model="R-GCN", variant=vname)
            comparison_rows.append(rr)
    if "ctx_results" in globals():
        for vname, oof in ctx_results.get(plan, {}).items():
            rr = summarize(oof, y, vname)
            rr.update(dataset=dataset_name, cv=cv_name, model="R-GCN", variant=vname)
            comparison_rows.append(rr)
    for vname, oof in GAT_RES[plan].items():
        rr = summarize(oof, y, vname)
        rr.update(dataset=dataset_name, cv=cv_name, model="GAT", variant=vname)
        comparison_rows.append(rr)
if comparison_rows:
    gat_comparison_df = pd.DataFrame(comparison_rows)
    gat_comparison_path = "results/rgcn_vs_gat_matched_comparison.csv"
    gat_comparison_df.to_csv(gat_comparison_path, index=False)
    print("Saved matched comparison:", gat_comparison_path)


Running these nine matched GAT variants:
 - GAT SBERT raw (base)
 - GAT SBERT collapsed (base)
 - GAT SBERT collapsed (reg)
 - GAT BERT-iso raw (base)
 - GAT BERT-iso collapsed (base)
 - GAT BERT-iso collapsed (reg)
 - GAT BERT-ctx raw (base)
 - GAT BERT-ctx collapsed (base)
 - GAT BERT-ctx collapsed (reg)
InDiBias /grouped    GAT SBERT raw (base)             acc=0.744 macro-F1=0.434 log-loss=0.558 (103s)
InDiBias /grouped    GAT SBERT collapsed (base)       acc=0.744 macro-F1=0.434 log-loss=0.558 (103s)
InDiBias /grouped    GAT SBERT collapsed (reg)        acc=0.745 macro-F1=0.427 log-loss=0.544 (60s)
InDiBias /grouped    GAT BERT-iso raw (base)          acc=0.749 macro-F1=0.444 log-loss=0.552 (109s)
InDiBias /grouped    GAT BERT-iso collapsed (base)    acc=0.749 macro-F1=0.444 log-loss=0.552 (109s)
InDiBias /grouped    GAT BERT-iso collapsed (reg)     acc=0.745 macro-F1=0.427 log-loss=0.542 (64s)
InDiBias /grouped    GAT BERT-ctx raw (base)          acc=0.746 macro-F1=0.431 log-loss=

KeyError: "['variant'] not in index"

In [ ]:
# Save and display the nine GAT variants
gat_df = pd.DataFrame(gat_rows)

gat_path = "results/gat_nine_variants_results.csv"
gat_df.to_csv(gat_path, index=False)

print(f"\nGAT evaluation completed in {(time.time()-t_gat_all)/60:.1f} min")
print("Saved:", gat_path)

display(
    gat_df[
        ["dataset", "cv", "method", "acc", "acc_sd", "macro_f1", "log_loss"]
    ].round(4)
)


# Optional matched comparison CSV using existing OOF predictions.
# This only summarizes existing predictions; it does not retrain either model.

comparison_rows = []

for plan in EVAL_PLAN:
    dataset_name, cv_name = plan
    y = DATA[dataset_name]["labels"]

    if "RES" in globals():
        for vname, oof in RES.get(plan, {}).items():
            rr = summarize(oof, y, vname)
            rr.update(
                dataset=dataset_name,
                cv=cv_name,
                model="R-GCN",
                method=vname
            )
            comparison_rows.append(rr)

    if "ctx_results" in globals():
        for vname, oof in ctx_results.get(plan, {}).items():
            rr = summarize(oof, y, vname)
            rr.update(
                dataset=dataset_name,
                cv=cv_name,
                model="R-GCN",
                method=vname
            )
            comparison_rows.append(rr)

    for vname, oof in GAT_RES[plan].items():
        rr = summarize(oof, y, vname)
        rr.update(
            dataset=dataset_name,
            cv=cv_name,
            model="GAT",
            method=vname
        )
        comparison_rows.append(rr)


if comparison_rows:
    gat_comparison_df = pd.DataFrame(comparison_rows)

    gat_comparison_path = "results/rgcn_vs_gat_matched_comparison.csv"
    gat_comparison_df.to_csv(gat_comparison_path, index=False)

    print("Saved matched comparison:", gat_comparison_path)

    display(
        gat_comparison_df[
            ["dataset", "cv", "model", "method",
             "acc", "acc_sd", "macro_f1", "log_loss"]
        ].round(4)
    )
else:
    print(
        "No matched comparison was generated. "
        "The required existing OOF prediction arrays may not be in memory."
    )


GAT evaluation completed in 77.5 min
Saved: results/gat_nine_variants_results.csv


,dataset,cv,method,acc,acc_sd,macro_f1,log_loss
0,InDiBias,grouped,GAT SBERT raw (base),0.7444,0.0063,0.4344,0.5576
1,InDiBias,grouped,GAT SBERT collapsed (base),0.7444,0.0063,0.4344,0.5576
2,InDiBias,grouped,GAT SBERT collapsed (reg),0.7454,0.0000,0.4271,0.5437
3,InDiBias,grouped,GAT BERT-iso raw (base),0.7495,0.0038,0.4437,0.5520
4,InDiBias,grouped,GAT BERT-iso collapsed (base),0.7495,0.0038,0.4437,0.5520
5,InDiBias,grouped,GAT BERT-iso collapsed (reg),0.7454,0.0000,0.4271,0.5421
6,InDiBias,grouped,GAT BERT-ctx raw (base),0.7464,0.0014,0.4313,0.5619
7,InDiBias,grouped,GAT BERT-ctx collapsed (base),0.7464,0.0014,0.4313,0.5619
8,InDiBias,grouped,GAT BERT-ctx collapsed (reg),0.7464,0.0014,0.4313,0.5552
9,InDiBias,stratified,GAT SBERT raw (base),0.7393,0.0075,0.4433,0.5628


Saved matched comparison: results/rgcn_vs_gat_matched_comparison.csv


,dataset,cv,model,method,acc,acc_sd,macro_f1,log_loss
0,InDiBias,grouped,R-GCN,R-GCN SBERT raw (base),0.7454,0.0025,0.4309,0.5626
1,InDiBias,grouped,R-GCN,R-GCN SBERT collapsed (base),0.7454,0.0025,0.4309,0.5626
2,InDiBias,grouped,R-GCN,R-GCN SBERT collapsed (reg),0.7454,0.0000,0.4271,0.5540
3,InDiBias,grouped,R-GCN,R-GCN BERT-iso raw (base),0.7526,0.0038,0.4560,0.5513
4,InDiBias,grouped,R-GCN,R-GCN BERT-iso collapsed (base),0.7526,0.0038,0.4560,0.5513
5,InDiBias,grouped,R-GCN,R-GCN BERT-iso collapsed (reg),0.7454,0.0000,0.4271,0.5542
6,InDiBias,grouped,R-GCN,R-GCN BERT-ctx raw (base),0.7495,0.0014,0.4438,0.5620
7,InDiBias,grouped,R-GCN,R-GCN BERT-ctx collapsed (base),0.7495,0.0014,0.4438,0.5620
8,InDiBias,grouped,R-GCN,R-GCN BERT-ctx collapsed (reg),0.7454,0.0000,0.4271,0.5608
9,InDiBias,grouped,GAT,GAT SBERT raw (base),0.7444,0.0063,0.4344,0.5576
